## TRATAMENTO DA BASE DE DADOS

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('vendas_aguas_LGPD.csv', delimiter=';')

df.head(10)

,data_pedido_entrega,semana,qnt_aguas_entregues,qnt_aguas_tickets,Obs,id_cliente
0,2023-01-02 00:00:00,Segunda,1.0,03/03,NaN,89
1,2023-01-02 00:00:00,Segunda,1.0,NaN,NaN,102
2,2023-01-02 00:00:00,Segunda,2.0,NaN,NaN,120
3,2023-01-02 00:00:00,Segunda,2.0,NaN,NaN,50
4,2023-01-02 00:00:00,Segunda,1.0,07/10,NaN,8
5,2023-01-02 00:00:00,Segunda,1.0,NaN,NaN,44
6,2023-01-04 00:00:00,Quarta,1.0,10/10,NaN,4
7,2023-01-04 00:00:00,Quarta,2.0,NaN,NaN,98
8,2023-01-04 00:00:00,Quarta,2.0,NaN,NaN,97
9,2023-01-05 00:00:00,Quinta,4.0,11/15,NaN,10


#### O que realizar na base:
- **Verificar os tipos de dados das colunas (data, str, int, ...), se precisar, formatar para o padrão certo;**
- Criar uma coluna somente com os anos (yyyy);
- Criar uma coluna somente com os meses (mm);
- Buscar identificar quais são os meses de cada estação e criar uma coluna com eles; 
- Desenvolver colunas como dias desde a última compra, a frequência de compra, o intervalo médio entre pedidos, a média de águas adquiridas por clientes, o total de pedidos realizados e a informação sobre a próxima compra;
- **Verificar valores ausentes e nulos e tráta-los;** <--------
- Padronizar o nome das colunas e a númeração do ID, colocar 4 digitos como padrão.

### 1 - ETAPA: Tipos de dados

In [4]:
# Conferindo os tipos de dados das colunas
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3137 entries, 0 to 3136
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   data_pedido_entrega  3137 non-null   object 
 1   semana               3137 non-null   object 
 2   qnt_aguas_entregues  3133 non-null   float64
 3   qnt_aguas_tickets    561 non-null    object 
 4   Obs                  24 non-null     object 
 5   id_cliente           3137 non-null   int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 147.2+ KB


**MODIFICAÇÃO DOS TIPO DE DADOS DAS COLUNAS**
- data_pedido_entrega = data
- qnt_aguas_entregues = int

In [ ]:
# Foi notado um erro em um ano na execução do código para converter a data, onde consta 12/07/0203, 
# ao análisar a base de dados é possível constatar que os dias, meses e anos seguem sequência, então conclui-se que o ano correto é 2023,
# optei por realizar a correção do valor para não ocorrer mais erros e não acarretar em perda dados
df['data_pedido_entrega'] = df['data_pedido_entrega'].astype(str).replace('/0203', '/2023')

# Modificação dos tipos de dados
df['data_pedido_entrega'] = pd.to_datetime(df['data_pedido_entrega'], format='mixed', dayfirst=True, errors='coerce')
df['qnt_aguas_entregues'] = df['qnt_aguas_entregues'].fillna(0).astype(int)

In [17]:
# Conferindo os tipos de dados das colunas após alteração
df.dtypes

data_pedido_entrega    datetime64[ns]
semana                         object
qnt_aguas_entregues             int64
qnt_aguas_tickets              object
Obs                            object
id_cliente                      int64
dtype: object

### 2 - ETAPA: Criar novas colunas para Insights e ML

In [18]:
# Coluna somente com os anos
df['ano'] = df['data_pedido_entrega'].dt.year

# Coluna somente com os meses
df['mes'] = df['data_pedido_entrega'].dt.month

Levando em consideração que a base de dados se refere a um depósito de água localizado na região Nordeste do Brasil, é importante considerar as características climáticas locais. Conforme destacado pelo Google, **"Embora as datas astronômicas sejam iguais para todo o Brasil, o Nordeste vivencia o clima de forma distinta, com clima quente predominando a maior parte do ano."**

Dessa forma, para tornar a análise mais aderente à realidade da região, as estações do ano foram definidas com base no regime climático local. Assim, o período de maior ocorrência de chuvas, geralmente entre os meses de **abril e agosto**, será classificado como **Inverno**, enquanto os demais meses, caracterizados por condições predominantemente secas e ensolaradas, serão classificados como **Verão**. Essa adaptação busca representar de forma mais fiel as variações climáticas que podem influenciar o comportamento dos dados analisados.

In [19]:
# Coluna das estações
def mapear_estacao(mes):
    if mes in [4, 5, 6, 7, 8]:
        return 'Inverno'
    else:
        return 'Verão'

df['estacao'] = df['mes'].apply(mapear_estacao)

In [ ]:
# Coluna com os dias dês de o último pedido/compra
df['dias_desde_utlima_compra']= (
    df.groupby('id_cliente')['data_pedido_entrega']
    .diff()
    .dt.days
)

In [ ]:
# Coluna de intervalo médio entre pedido/compra por cliente
df['intervalo_medio'] = (
    df.groupby('id_cliente')['dias_desde_utlima_compra']
    .transform('mean')
)

In [ ]:
# Coluna de frêquencia de pedido/compra por cliente
df['frequencia_compra'] = (
    df.groupby('id_cliente')
    .cumcount() + 1
)

In [ ]:
# Coluna constando a quantidade média de águas por cliente
df['media_aguas_cliente'] = (
    df.groupby('id_cliente')['qnt_aguas_entregues']
    .transform('mean')
)

In [ ]:
# Coluna de total de pedidos por cliente
df['total_pedidos'] = (
    df.groupby('id_cliente')
    .cumcount() + 1
)

In [32]:
# Colunas de probabilidade de compra futura
df['proxima_compra'] = (
    df.groupby('id_cliente')['data_pedido_entrega']
    .shift(-1)
)

df['dias_ate_proxima_compra'] = (
    df['proxima_compra'] - df['data_pedido_entrega']
).dt.days

df['vai_comprar_15dias'] = np.where(
    df['dias_ate_proxima_compra'] <= 15,
    1, 
    0
)

In [49]:
# Exibição do DataFrame com as novas colunas
df

,data_pedido_entrega,semana,qnt_aguas_entregues,qnt_aguas_tickets,Obs,id_cliente,ano,mes,estacao,dias_desde_utlima_compra,frequencia_compra,intervalo_medio,media_aguas_cliente,total_pedidos,proxima_compra,dias_ate_proxima_compra,vai_comprar_15dias
0,2023-01-02,Segunda,1,03/03,NaN,89,2023.0,1.0,Verão,0,1,5.91,1.01,1,2023-01-11,9,1
1,2023-01-02,Segunda,1,NaN,NaN,102,2023.0,1.0,Verão,0,1,NaN,1.00,1,NaT,0,0
2,2023-01-02,Segunda,2,NaN,NaN,120,2023.0,1.0,Verão,0,1,14.07,1.93,1,2023-01-18,16,0
3,2023-01-02,Segunda,2,NaN,NaN,50,2023.0,1.0,Verão,0,1,7.80,2.03,1,2023-01-09,7,1
4,2023-01-02,Segunda,1,07/10,NaN,8,2023.0,1.0,Verão,0,1,6.15,1.00,1,2023-01-06,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3132,2025-08-20,Quarta,1,NaN,NaN,24,2025.0,8.0,Inverno,72,19,28.39,1.16,19,NaT,0,0
3133,2025-08-20,Quarta,2,NaN,NaN,72,2025.0,8.0,Inverno,31,31,16.10,1.52,31,NaT,0,0
3134,2025-08-20,Quarta,1,NaN,NaN,134,2025.0,8.0,Inverno,22,41,23.15,2.05,41,NaT,0,0
3135,2025-08-21,Quinta,1,NaN,NaN,118,2025.0,8.0,Inverno,16,93,10.34,1.01,93,NaT,0,0


In [ ]:
# Formatando algumas novas colunas
# Nenhuma casa decimal
colunas2 = [
    'dias_desde_utlima_compra',
    'total_pedidos',
    'dias_ate_proxima_compra'
]

df[colunas2] = df[colunas2].fillna(0).astype(int)

# 2 casas decimais
colunas1 = [
    'intervalo_medio',
    'media_aguas_cliente'
]

df[colunas1] = df[colunas1].round(2)

### 3 - ETAPA: Análise e tratamento de valores nulos e Outlliers

In [6]:
# Verificação de valores nulos
df.isnull().sum()


data_pedido_entrega       0
semana                    0
qnt_aguas_entregues       4
qnt_aguas_tickets      2576
Obs                    3113
id_cliente                0
dtype: int64

In [8]:
# Verificação de Outlliers
df.describe()

,qnt_aguas_entregues,id_cliente
count,3133.000000,3137.000000
mean,1.365464,65.095314
std,0.563509,42.983491
min,1.000000,1.000000
25%,1.000000,25.000000
50%,1.000000,56.000000
75%,2.000000,99.000000
max,8.000000,136.000000
